# Aprendizado de máquina aplicado à biologia

Notebook único do workshop. As discussões estão nos slides; aqui fica o código, na ordem da aula.
**Execute as células de cima para baixo**: cada parte usa variáveis criadas nas células anteriores.

| Parte | Tema | *Dataset* |
|---|---|---|
| 1 | Regressão linear | *Diabetes* (442 pacientes) |
| 2 | Classificação | *Breast Cancer Wisconsin* (569 nódulos) |
| 3 | Métodos não supervisionados | *Mice Protein Expression* (72 camundongos) |

## 0. Preparação do ambiente

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42  # semente usada em todas as divisões e modelos, para resultados reprodutíveis

In [ ]:
# Configuração visual dos gráficos (paleta do material)
from cycler import cycler
from matplotlib.colors import LinearSegmentedColormap

CORES = {
    "papel": "#f8f4ea",
    "texto": "#332d28",
    "texto_secundario": "#62584f",
    "linha": "#d9cfbe",
    "azul": "#435d73",
    "verde": "#2d4038",
    "ocre": "#90682e",
    "azul_acinzentado": "#6b7c90",
    "preenchimento_azul": "#e9eef2",
    "preenchimento_creme": "#f4e6cd",
}

CMAP_CORRELACAO = LinearSegmentedColormap.from_list(
    "azul_papel_ocre", [CORES["azul"], CORES["papel"], CORES["ocre"]]
)
CMAP_CONTAGEM = LinearSegmentedColormap.from_list("papel_azul", [CORES["papel"], CORES["azul"]])

ESTILO_PONTOS = {"alpha": 0.6, "edgecolor": CORES["papel"], "linewidth": 0.5}

ESTILO_BOXPLOT = {
    "color": CORES["preenchimento_azul"],
    "linecolor": CORES["azul"],
    "medianprops": {"color": CORES["azul"], "linewidth": 2},
    "showfliers": False,
}

plt.rcParams.update({
    "figure.figsize": (6, 4),
    "figure.dpi": 110,
    "figure.facecolor": CORES["papel"],
    "savefig.facecolor": CORES["papel"],
    "axes.facecolor": CORES["papel"],
    "axes.edgecolor": CORES["linha"],
    "axes.labelcolor": CORES["texto"],
    "axes.titlecolor": CORES["texto"],
    "axes.prop_cycle": cycler(color=[
        CORES["azul"], CORES["verde"], CORES["ocre"], CORES["azul_acinzentado"]
    ]),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "axes.axisbelow": True,
    "grid.color": CORES["linha"],
    "grid.linewidth": 0.8,
    "patch.edgecolor": CORES["papel"],
    "lines.linewidth": 2,
    "text.color": CORES["texto"],
    "xtick.color": CORES["texto_secundario"],
    "ytick.color": CORES["texto_secundario"],
    "legend.facecolor": CORES["papel"],
    "legend.edgecolor": CORES["linha"],
    "legend.framealpha": 1,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 13,
})

# Parte 1 — Regressão linear: progressão do diabetes

**Pergunta central:** o modelo consegue prever uma medida da progressão do diabetes, um ano após a avaliação inicial, a partir das medidas dos pacientes?

## 1.1 Carregando os dados

In [ ]:
from sklearn.datasets import load_diabetes

dataset = load_diabetes(as_frame=True, scaled=False)

X = dataset.data                                     # features: uma linha por paciente
y = dataset.target.rename("disease_progression")    # target: progressão um ano depois

## 1.2 Exploração inicial

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
X.info()

In [ ]:
# sex está armazenado como número, mas representa duas categorias
X["sex"].value_counts(normalize=True) * 100

In [ ]:
X_cat_cols = ["sex"]
X_num_cols = [col for col in X.columns if col not in X_cat_cols]

print("Numéricas:", X_num_cols)
print("Categóricas:", X_cat_cols)

## 1.3 Ferramentas estatísticas

### Resumo estatístico

In [ ]:
X[X_num_cols].describe()

In [ ]:
y.describe()

### Valor típico: média, mediana e moda

In [ ]:
pd.DataFrame({
    "Média": [y.mean()],
    "Mediana": [y.median()],
    "Modas": [y.mode().tolist()],
}).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(y.index, y, s=18, color=CORES["azul"], **ESTILO_PONTOS)
ax.axhline(y.median(), color=CORES["verde"], label=f"Mediana = {y.median():.1f}")
for i, moda in enumerate(y.mode()):
    ax.axhline(moda, color=CORES["azul_acinzentado"], linestyle=":", label="Moda" if i == 0 else None)
ax.scatter(y.index.values.mean(), y.mean(), marker="x", s=160, linewidths=3,
           color=CORES["ocre"], zorder=3, label=f"Média = {y.mean():.1f}")
ax.set(xlabel="Paciente", ylabel="Progressão da doença")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1), ncols=3, frameon=False)
fig.tight_layout()
plt.show()

### Dispersão: desvio padrão e *boxplot*

In [ ]:
media, desvio = y.mean(), y.std()
q1, q2, q3 = y.quantile([0.25, 0.5, 0.75])

fig, (ax_pontos, ax_box) = plt.subplots(1, 2, sharey=True, figsize=(8, 4.5), width_ratios=[3, 1])
ax_pontos.axhspan(media - desvio, media + desvio, color=CORES["preenchimento_creme"], zorder=0,
                  label=f"Média ± desvio padrão ({desvio:.1f})")
ax_pontos.scatter(y.index, y, s=18, color=CORES["azul"], **ESTILO_PONTOS)
ax_pontos.axhline(media, color=CORES["ocre"], linestyle="--")
ax_pontos.set(xlabel="Paciente", ylabel="Progressão da doença")
ax_pontos.legend(loc="lower center", bbox_to_anchor=(0.5, 1), frameon=False)

sns.boxplot(y=y, width=0.4, ax=ax_box, **ESTILO_BOXPLOT)
for nome, valor in [("Q1", q1), ("Q2", q2), ("Q3", q3)]:
    ax_box.annotate(f"{nome} = {valor:.1f}", (0.2, valor), xytext=(6, 0), textcoords="offset points", va="center")
ax_box.set(title=f"IQR = {q3 - q1:.1f}", ylabel="", xticks=[])
ax_box.grid(False)
fig.tight_layout()
plt.show()

### Forma da distribuição: histograma, probabilidade e KDE

In [ ]:
limites = [0, 50, 100, 150, 200, 250, 300, 350]
caixas = pd.cut(y, bins=limites, right=False)

pd.DataFrame({
    "Pacientes": caixas.value_counts(sort=False),
    "Frequência relativa": caixas.value_counts(normalize=True, sort=False).round(4),
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.histplot(y, bins=limites, color=CORES["azul"], alpha=0.75, ax=axes[0])
axes[0].set(title="Frequência absoluta", xlabel="Progressão da doença", ylabel="Pacientes")

sns.histplot(y, bins=limites, stat="density", color=CORES["azul"], alpha=0.75, label="Histograma", ax=axes[1])
sns.kdeplot(y, cut=0, color=CORES["verde"], label="KDE", ax=axes[1])
axes[1].set(title="Densidade (área = probabilidade)", xlabel="Progressão da doença", ylabel="Densidade")
axes[1].legend()
fig.tight_layout()
plt.show()

### Relações entre variáveis: correlação de Pearson

In [ ]:
r_bmi = X["bmi"].corr(y)
inclinacao, intercepto = np.polyfit(X["bmi"], y, deg=1)
x_reta = np.array([X["bmi"].min(), X["bmi"].max()])

fig, ax = plt.subplots()
ax.scatter(X["bmi"], y, s=18, color=CORES["azul"], **ESTILO_PONTOS)
ax.plot(x_reta, intercepto + inclinacao * x_reta, color=CORES["ocre"], linestyle="--", label="Tendência linear")
ax.set(title=f"r = {r_bmi:.2f}", xlabel="Índice de massa corporal", ylabel="Progressão da doença")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
corr = X[X_num_cols].join(y).corr()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), width_ratios=[1, 1.4])
corr_target = corr[y.name].drop(y.name).sort_values()
axes[0].barh(corr_target.index, corr_target.values, color=CORES["azul"])
axes[0].axvline(0, color=CORES["texto_secundario"], linewidth=0.8)
axes[0].grid(False)
axes[0].grid(axis="x")
axes[0].set(title="Correlação com a progressão", xlabel="r de Pearson")

sns.heatmap(corr, annot=True, fmt=".2f", annot_kws={"size": 8}, cmap=CMAP_CORRELACAO,
            vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Matriz de correlação")
fig.tight_layout()
plt.show()

In [ ]:
# A única feature categórica: progressão por grupo de sexo
y.groupby(X["sex"]).agg(["mean", "median", "std", "count"]).round(2)

## 1.4 Divisão do *dataset*

Célula de referência: treino (60%), validação (20%) e teste (20%). O teste só é usado no fim.

In [ ]:
from sklearn.model_selection import train_test_split

X_full_train, X_test, y_full_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_full_train, y_full_train, test_size=0.25, random_state=SEED)

print(f"Treino: {len(X_train)} | Validação: {len(X_val)} | Teste: {len(X_test)}")

## 1.5 Regressão linear simples (uma *feature*: IMC)

In [ ]:
# Colchetes duplos: o Scikit-Learn espera uma tabela, mesmo com uma só feature
X_train_bmi = X_train[["bmi"]]
X_val_bmi = X_val[["bmi"]]

X_train_bmi.shape, X_val_bmi.shape

### *Baseline*: prever sempre a média do treino

In [ ]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train_bmi, y_train)
baseline_pred = baseline.predict(X_val_bmi)

baseline_pred[:5]

### Ajustando a reta

In [ ]:
from sklearn.linear_model import LinearRegression

simple_model = LinearRegression()
simple_model.fit(X_train_bmi, y_train)

w0 = simple_model.intercept_
w1 = simple_model.coef_[0]
print(f"Intercepto (w0): {w0:.2f}")
print(f"Inclinação (w1): {w1:.2f}")

In [ ]:
x_reta = np.linspace(X_train["bmi"].min(), X_train["bmi"].max(), 2)

fig, ax = plt.subplots()
ax.scatter(X_train["bmi"], y_train, s=18, color=CORES["azul"], label="Pacientes de treino", **ESTILO_PONTOS)
ax.plot(x_reta, w0 + w1 * x_reta, color=CORES["ocre"], label=f"Reta: ŷ = {w0:.1f} + {w1:.2f}·IMC")
ax.axhline(y_train.mean(), color=CORES["texto_secundario"], linestyle="--", label="Baseline")
ax.set(xlabel="Índice de massa corporal", ylabel="Progressão da doença")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
plt.show()

### Previsões e resíduos na validação

In [ ]:
simple_pred = simple_model.predict(X_val_bmi)
residuos = y_val - simple_pred      # e_i = y_i - ŷ_i

x_reta = np.linspace(X_val["bmi"].min(), X_val["bmi"].max(), 2)
fig, ax = plt.subplots()
ax.vlines(X_val["bmi"], simple_pred, y_val, color=CORES["azul_acinzentado"], linewidth=1, alpha=0.6, label="Resíduo")
ax.scatter(X_val["bmi"], y_val, s=18, color=CORES["azul"], label="Pacientes de validação", **ESTILO_PONTOS)
ax.plot(x_reta, w0 + w1 * x_reta, color=CORES["ocre"], label="Reta ajustada no treino")
ax.set(xlabel="Índice de massa corporal", ylabel="Progressão da doença")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
plt.show()

### RMSE

In [ ]:
from sklearn.metrics import root_mean_squared_error

baseline_rmse = root_mean_squared_error(y_val, baseline_pred)
simple_rmse = root_mean_squared_error(y_val, simple_pred)

print(f"RMSE do baseline: {baseline_rmse:.2f}")
print(f"RMSE da regressão simples: {simple_rmse:.2f}")

### Análise dos resíduos

In [ ]:
fig, ax = plt.subplots()
ax.scatter(simple_pred, residuos, s=18, color=CORES["azul"], **ESTILO_PONTOS)
ax.axhline(0, color=CORES["texto_secundario"], linestyle="--", label="Resíduo = 0")
ax.axhline(residuos.mean(), color=CORES["ocre"], label=f"Média dos resíduos = {residuos.mean():.2f}")
ax.set(xlabel="Progressão prevista", ylabel="Resíduo")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

residuos_train = y_train - simple_model.predict(X_train_bmi)
print(f"Média dos resíduos no treino: {residuos_train.mean():.2e}")
print(f"Média dos resíduos na validação: {residuos.mean():.2f}")

## 1.6 Regressão linear múltipla (todas as *features*)

In [ ]:
multiple_model = LinearRegression()
multiple_model.fit(X_train, y_train)
multiple_pred = multiple_model.predict(X_val)

# Um peso por feature, além do intercepto
pd.Series(multiple_model.coef_, index=X_train.columns, name="peso").round(2)

In [ ]:
def resumo_validacao(previsao):
    erro = y_val - previsao
    erro_baseline = y_val - baseline_pred
    return {
        "RMSE na validação": root_mean_squared_error(y_val, previsao),
        "Variância explicada": 1 - (erro**2).sum() / (erro_baseline**2).sum(),
        "Média dos resíduos": erro.mean(),
    }


pd.DataFrame({
    "Baseline": resumo_validacao(baseline_pred),
    "Regressão simples": resumo_validacao(simple_pred),
    "Regressão múltipla": resumo_validacao(multiple_pred),
}).T.round(2)

In [ ]:
lims = [y_val.min(), y_val.max()]
fig, ax = plt.subplots()
ax.scatter(y_val, multiple_pred, s=22, color=CORES["verde"], label="Pacientes de validação", **ESTILO_PONTOS)
ax.plot(lims, lims, color=CORES["texto_secundario"], linestyle="--", label="Previsão = observado")
ax.set(xlabel="Progressão observada", ylabel="Progressão prevista")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 1.7 Validação cruzada

In [ ]:
# Uma única divisão é uma única fotografia: outras sementes, outros resultados
linhas = []
for semente in range(6):
    Xa, Xb, ya, yb = train_test_split(X_full_train, y_full_train, test_size=0.25, random_state=semente)
    modelo = LinearRegression().fit(Xa[["bmi"]], ya)
    erro = yb - modelo.predict(Xb[["bmi"]])
    linhas.append({"Semente": semente, "Média dos resíduos": erro.mean(), "RMSE": np.sqrt((erro**2).mean())})

pd.DataFrame(linhas).round(1)

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

modelos_cv = {
    "Baseline": (DummyRegressor(strategy="mean"), X_full_train),
    "Regressão simples": (LinearRegression(), X_full_train[["bmi"]]),
    "Regressão múltipla": (LinearRegression(), X_full_train),
}

# O Scikit-Learn devolve o RMSE com sinal negativo (convenção "maior é melhor"); invertemos o sinal
rmse_cv = {
    nome: -cross_val_score(modelo, X_cv, y_full_train, cv=kf, scoring="neg_root_mean_squared_error")
    for nome, (modelo, X_cv) in modelos_cv.items()
}

pd.DataFrame({nome: {"Média": s.mean(), "Desvio padrão": s.std()} for nome, s in rmse_cv.items()}).T.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
for posicao, (nome, valores) in enumerate(rmse_cv.items()):
    ax.scatter(valores, [posicao] * len(valores), s=40, color=CORES["azul"], **ESTILO_PONTOS)
    ax.scatter(valores.mean(), posicao, marker="x", s=140, linewidths=3, color=CORES["ocre"], zorder=3)
ax.set_yticks(range(len(rmse_cv)), list(rmse_cv))
ax.grid(False)
ax.grid(axis="x")
ax.set(xlabel="RMSE na validação cruzada (5 dobras; x = média)")
fig.tight_layout()
plt.show()

## 1.8 Avaliação final no conjunto de teste

In [ ]:
# Modelo escolhido, retreinado com todo o full_train, avaliado uma única vez no teste
final_model = LinearRegression().fit(X_full_train, y_full_train)
final_pred = final_model.predict(X_test)

baseline_final = DummyRegressor(strategy="mean").fit(X_full_train, y_full_train)

print(f"RMSE no teste (regressão múltipla): {root_mean_squared_error(y_test, final_pred):.2f}")
print(f"RMSE no teste (baseline): {root_mean_squared_error(y_test, baseline_final.predict(X_test)):.2f}")

In [ ]:
lims = [y_test.min(), y_test.max()]
fig, ax = plt.subplots()
ax.scatter(y_test, final_pred, s=22, color=CORES["verde"], label="Pacientes de teste", **ESTILO_PONTOS)
ax.plot(lims, lims, color=CORES["texto_secundario"], linestyle="--", label="Previsão = observado")
ax.set(xlabel="Progressão observada", ylabel="Progressão prevista")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

# Parte 2 — Classificação: nódulos de mama

**Pergunta central:** as medidas dos núcleos celulares distinguem nódulos benignos de malignos?

## 2.1 Carregando os dados

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer(as_frame=True)

X_bc = cancer.data
# No dataset, 0 = maligno e 1 = benigno; usamos os nomes para evitar confusão
y_bc = cancer.target.map({0: "maligno", 1: "benigno"}).rename("diagnostico")

X_bc.shape

## 2.2 Exploração inicial

In [ ]:
X_bc.head()

In [ ]:
print(f"Valores ausentes: {X_bc.isna().sum().sum()}")
y_bc.value_counts().to_frame("nódulos").assign(proporcao=lambda t: (t["nódulos"] / t["nódulos"].sum()).round(3))

In [ ]:
cores_diagnostico = {"benigno": CORES["azul"], "maligno": CORES["ocre"]}
exemplos_bc = ["mean radius", "mean texture", "mean concave points", "mean smoothness"]

fig, axes = plt.subplots(1, 4, figsize=(13, 3.8))
for ax, feature in zip(axes, exemplos_bc):
    sns.boxplot(x=y_bc, y=X_bc[feature], order=["benigno", "maligno"], width=0.5, ax=ax, **ESTILO_BOXPLOT)
    sns.stripplot(x=y_bc, y=X_bc[feature], order=["benigno", "maligno"], hue=y_bc, palette=cores_diagnostico,
                  legend=False, size=2.5, alpha=0.5, ax=ax)
    ax.set(title=feature, xlabel="", ylabel="")
fig.tight_layout()
plt.show()

In [ ]:
# Muitas features carregam informação parecida (ex.: raio, perímetro e área)
fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(X_bc.corr(), cmap=CMAP_CORRELACAO, vmin=-1, vmax=1, xticklabels=True, yticklabels=True, ax=ax)
ax.tick_params(labelsize=7)
ax.set_title("Correlação entre as 30 features")
fig.tight_layout()
plt.show()

## 2.3 Divisão estratificada do *dataset*

Célula de referência. `stratify` preserva a proporção de benignos e malignos em cada parte.

In [ ]:
X_bc_full_train, X_bc_test, y_bc_full_train, y_bc_test = train_test_split(
    X_bc, y_bc, test_size=0.2, stratify=y_bc, random_state=SEED
)
X_bc_train, X_bc_val, y_bc_train, y_bc_val = train_test_split(
    X_bc_full_train, y_bc_full_train, test_size=0.25, stratify=y_bc_full_train, random_state=SEED
)

pd.DataFrame({
    nome: partes.value_counts(normalize=True)
    for nome, partes in [("Completo", y_bc), ("Treino", y_bc_train), ("Validação", y_bc_val), ("Teste", y_bc_test)]
}).round(3)

## 2.4 *Baseline*: prever sempre a classe mais frequente

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline_bc = DummyClassifier(strategy="most_frequent")
baseline_bc.fit(X_bc_train, y_bc_train)
baseline_bc_pred = baseline_bc.predict(X_bc_val)

print(f"Acurácia do baseline: {accuracy_score(y_bc_val, baseline_bc_pred):.3f}")

## 2.5 Regressão logística

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Padronização + regressão logística em um único objeto (pipeline)
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
logistic.fit(X_bc_train, y_bc_train)
logistic_pred = logistic.predict(X_bc_val)

In [ ]:
# A regressão logística devolve uma probabilidade para cada classe; predict() usa o limiar de 0,5
probabilidades = pd.DataFrame(logistic.predict_proba(X_bc_val), columns=logistic.classes_).round(3)
probabilidades.assign(previsto=logistic_pred, observado=y_bc_val.values).head()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report


def matriz_confusao(observado, previsto, titulo):
    fig, ax = plt.subplots(figsize=(4.2, 3.8))
    ConfusionMatrixDisplay.from_predictions(
        observado, previsto, labels=["maligno", "benigno"], cmap=CMAP_CONTAGEM, colorbar=False, ax=ax
    )
    ax.grid(False)
    ax.set(title=titulo, xlabel="Previsto", ylabel="Observado")
    fig.tight_layout()
    plt.show()


print(f"Acurácia: {accuracy_score(y_bc_val, logistic_pred):.3f}")
matriz_confusao(y_bc_val, logistic_pred, "Regressão logística (validação)")
print(classification_report(y_bc_val, logistic_pred, digits=3))

## 2.6 Árvore de decisão

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree = DecisionTreeClassifier(max_depth=3, random_state=SEED)
tree.fit(X_bc_train, y_bc_train)
tree_pred = tree.predict(X_bc_val)

print(f"Acurácia: {accuracy_score(y_bc_val, tree_pred):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6.5))
plot_tree(tree, feature_names=X_bc.columns, class_names=tree.classes_, filled=False,
          impurity=False, proportion=True, rounded=True, fontsize=10, ax=ax)
fig.tight_layout()
plt.show()

In [ ]:
matriz_confusao(y_bc_val, tree_pred, "Árvore de decisão (validação)")

# Quanto cada feature contribuiu para as divisões da árvore
importancias = pd.Series(tree.feature_importances_, index=X_bc.columns, name="importância")
importancias[importancias > 0].sort_values(ascending=False).round(3)

## 2.7 Validação cruzada estratificada

In [ ]:
from sklearn.metrics import make_scorer, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
metricas = {
    "acurácia": "accuracy",
    "recall (maligno)": make_scorer(recall_score, pos_label="maligno"),
}

classificadores = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Regressão logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Árvore de decisão": DecisionTreeClassifier(max_depth=3, random_state=SEED),
}

linhas = {}
for nome, modelo in classificadores.items():
    resultado = cross_validate(modelo, X_bc_full_train, y_bc_full_train, cv=skf, scoring=metricas)
    linhas[nome] = {
        f"{metrica} ({estatistica})": funcao(resultado[f"test_{metrica}"])
        for metrica in metricas
        for estatistica, funcao in [("média", np.mean), ("dp", np.std)]
    }

pd.DataFrame(linhas).T.round(3)

## 2.8 Avaliação final no conjunto de teste

In [ ]:
# Modelo escolhido, retreinado com todo o full_train, avaliado uma única vez no teste
final_bc = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
final_bc.fit(X_bc_full_train, y_bc_full_train)
final_bc_pred = final_bc.predict(X_bc_test)

print(f"Acurácia no teste: {accuracy_score(y_bc_test, final_bc_pred):.3f}")
matriz_confusao(y_bc_test, final_bc_pred, "Regressão logística (teste)")
print(classification_report(y_bc_test, final_bc_pred, digits=3))

# Parte 3 — Métodos não supervisionados: proteínas do córtex de camundongos

**Pergunta central:** os perfis das 77 proteínas distinguem os grupos experimentais? Quais proteínas mais contribuem para as diferenças encontradas?

Os rótulos (genótipo, comportamento, tratamento) **não entram em nenhum ajuste**: servem só para interpretar os resultados.

## 3.1 Carregando os dados

In [ ]:
from pathlib import Path

# No computador local, lê a pasta data/; no Colab, lê a mesma cópia direto do GitHub
CAMINHO_LOCAL = Path("data/mice_protein_expression.csv")
URL_REPOSITORIO = "https://raw.githubusercontent.com/<usuario>/<repo>/main/data/mice_protein_expression.csv"

df_full = pd.read_csv(CAMINHO_LOCAL if CAMINHO_LOCAL.exists() else URL_REPOSITORIO)
df_full.shape

In [ ]:
# MouseID = <animal>_<medição>: 72 animais x 15 medições = 1080 linhas
partes_id = df_full["MouseID"].str.split("_", expand=True)
df = df_full.assign(animal_id=partes_id[0], measurement=pd.to_numeric(partes_id[1]))

print(f"Animais: {df['animal_id'].nunique()}")
print(f"Medições por animal: {len(df) // df['animal_id'].nunique()}")

In [ ]:
all_protein_cols = [col for col in df.columns if col.endswith("_N")]
meta_cols = ["Genotype", "Treatment", "Behavior", "class"]

# pS6_N é uma cópia exata de ARC_N: ficamos com 76 proteínas
print("pS6_N idêntica a ARC_N:", df["pS6_N"].equals(df["ARC_N"]))
protein_cols = [col for col in all_protein_cols if col != "pS6_N"]

len(all_protein_cols), len(protein_cols)

In [ ]:
df.groupby("class").agg(animais=("animal_id", "nunique"))

## 3.2 Unidade de análise: o animal, não a medição

In [ ]:
# within: desvio padrão das 15 medições de cada animal, com a média dos 72 desvios
# between: desvio padrão das 72 médias por animal
within = df.groupby("animal_id")[protein_cols].std().mean().rename("within")
between = df.groupby("animal_id")[protein_cols].mean().std().rename("between")

variabilidade = pd.concat([within, between], axis=1)
variabilidade["razao"] = variabilidade["within"] / variabilidade["between"]
variabilidade.sort_values("razao").round(3)

In [ ]:
abaixo = variabilidade["razao"] < 1

fig, ax = plt.subplots(figsize=(9, 3.2))
sns.swarmplot(x=variabilidade["razao"], hue=abaixo, palette={True: CORES["azul"], False: CORES["ocre"]},
              legend=False, size=6, ax=ax)
ax.axvline(1, color=CORES["texto_secundario"], linestyle="--")
ax.text(0.98, 0.92, f"{abaixo.sum()} proteínas abaixo de 1 | {(~abaixo).sum()} acima",
        transform=ax.transAxes, ha="right", color=CORES["texto_secundario"])
ax.grid(False)
ax.grid(axis="x")
ax.spines["left"].set_visible(False)
ax.set(xlabel="Razão: variação intra-animal / variação entre animais", yticks=[])
fig.tight_layout()
plt.show()

In [ ]:
# Cada animal passa a ser uma linha: a média das suas 15 medições
animal_keys = ["animal_id"] + meta_cols
animal_mean_df = df.groupby(animal_keys, as_index=False)[protein_cols].mean()

animal_mean_df.shape

## 3.3 Faltantes, imputação e padronização

In [ ]:
faltantes = animal_mean_df[protein_cols].isna().sum()
faltantes[faltantes > 0].sort_values(ascending=False).to_frame("animais sem média")

In [ ]:
from sklearn.impute import SimpleImputer

# Faltante -> mediana da proteína; depois, cada proteína com média 0 e desvio padrão 1
preparo = make_pipeline(SimpleImputer(strategy="median"), StandardScaler()).set_output(transform="pandas")
proteinas_preparadas = preparo.fit_transform(animal_mean_df[protein_cols])

exemplos = ["SOD1_N", "BCL2_N", "H3AcK18_N"]
pd.concat({
    "Antes": animal_mean_df[exemplos].agg(["mean", "std"]).T,
    "Depois": proteinas_preparadas[exemplos].agg(["mean", "std"]).T,
}, axis=1).round(3) + 0.0

## 3.4 Análise de componentes principais (PCA)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA()
componentes = pca.fit_transform(proteinas_preparadas)   # fit recebe só X: nenhum rótulo

pc_names = [f"PC{i + 1}" for i in range(pca.n_components_)]
scores = pd.DataFrame(componentes, columns=pc_names).join(animal_mean_df[meta_cols])
loadings = pd.DataFrame(pca.components_.T, index=protein_cols, columns=pc_names)
explicada = pd.Series(pca.explained_variance_ratio_, index=pc_names)

explicada.head(10).to_frame("Explicada").assign(Acumulada=lambda t: t["Explicada"].cumsum()).round(3)

In [ ]:
n = 15
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(explicada.index[:n], explicada.values[:n], color=CORES["azul"], label="Por componente")
ax.plot(explicada.index[:n], explicada.cumsum().values[:n], color=CORES["ocre"], marker="o", label="Acumulada")
ax.set(xlabel="Componente", ylabel="Fração da variância explicada", ylim=(0, 1))
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
cores_behavior = {"C/S": CORES["azul"], "S/C": CORES["ocre"]}
rotulo_pc1 = f"PC1 ({explicada['PC1']:.0%} da variância)"
rotulo_pc2 = f"PC2 ({explicada['PC2']:.0%} da variância)"

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
sns.scatterplot(data=scores, x="PC1", y="PC2", hue="Behavior", palette=cores_behavior, s=60, ax=axes[0],
                **{**ESTILO_PONTOS, "alpha": 0.85})
sns.scatterplot(data=scores, x="PC1", y="PC2", hue="Genotype", style="Treatment",
                palette=[CORES["verde"], CORES["azul_acinzentado"]], s=60, ax=axes[1],
                **{**ESTILO_PONTOS, "alpha": 0.85})
for ax, titulo in zip(axes, ["Colorido por comportamento", "Genótipo (cor) e tratamento (forma)"]):
    ax.axhline(0, color=CORES["linha"], linewidth=0.8, zorder=0)
    ax.axvline(0, color=CORES["linha"], linewidth=0.8, zorder=0)
    ax.set(title=titulo, xlabel=rotulo_pc1, ylabel=rotulo_pc2)
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
def top_loadings(pc, n=10):
    """Proteínas com maior |loading| numa componente, mantendo o sinal."""
    return loadings[pc].reindex(loadings[pc].abs().sort_values(ascending=False).index)[:n]


fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, pc in zip(axes, ["PC1", "PC2"]):
    top = top_loadings(pc)[::-1]
    ax.barh(top.index, top.values, color=np.where(top.values > 0, CORES["azul"], CORES["ocre"]))
    ax.axvline(0, color=CORES["texto_secundario"], linewidth=0.8)
    ax.grid(False)
    ax.grid(axis="x")
    ax.set(title=f"{pc}: 10 maiores |loadings|", xlabel="Loading")
fig.tight_layout()
plt.show()

In [ ]:
# Proteínas que mais pesam em PC1 e em PC2, comparadas entre os grupos de comportamento
proteinas_pcs = top_loadings("PC1", 3).index.tolist() + top_loadings("PC2", 3).index.tolist()

fig, axes = plt.subplots(1, len(proteinas_pcs), figsize=(14, 3.6))
for ax, proteina in zip(axes, proteinas_pcs):
    sns.boxplot(data=animal_mean_df, x="Behavior", y=proteina, order=list(cores_behavior), width=0.5, ax=ax,
                **ESTILO_BOXPLOT)
    sns.stripplot(data=animal_mean_df, x="Behavior", y=proteina, order=list(cores_behavior), hue="Behavior",
                  palette=cores_behavior, legend=False, size=3.5, alpha=0.7, ax=ax)
    pc = "PC1" if proteina in proteinas_pcs[:3] else "PC2"
    ax.set(title=f"{proteina}\n({pc})", xlabel="", ylabel="")
fig.tight_layout()
plt.show()

## 3.5 K-means

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

plano = scores[["PC1", "PC2"]]


def varrer_k(dados):
    """K-means para k = 2 a 10: silhueta (separação) e ARI contra Behavior (só para interpretar)."""
    linhas = []
    for k in range(2, 11):
        grupos = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit_predict(dados)
        linhas.append({
            "k": k,
            "silhueta": silhouette_score(dados, grupos),
            "ARI Behavior": adjusted_rand_score(scores["Behavior"], grupos),
        })
    return pd.DataFrame(linhas).set_index("k")


resultados_bruto = varrer_k(plano)
resultados_bruto.round(3)

In [ ]:
kmeans_2 = KMeans(n_clusters=2, n_init=20, random_state=SEED).fit(plano)
pd.crosstab(pd.Series(kmeans_2.labels_, name="grupo"), scores["Behavior"])

### Padronizando PC1 e PC2

In [ ]:
# O K-means usa distâncias: a componente de maior variância domina. Padronizando, as duas pesam igual
plano_padronizado = (plano - plano.mean()) / plano.std()

resultados_padronizado = varrer_k(plano_padronizado)
kmeans_2_pad = KMeans(n_clusters=2, n_init=20, random_state=SEED).fit(plano_padronizado)

display(resultados_padronizado.round(3))
pd.crosstab(pd.Series(kmeans_2_pad.labels_, name="grupo"), scores["Behavior"])

In [ ]:
cores_grupo = {0: CORES["azul"], 1: CORES["verde"]}
marcadores_grupo = {0: "o", 1: "s"}
paineis = [
    ("Plano original", plano, kmeans_2, rotulo_pc1, rotulo_pc2),
    ("Plano padronizado", plano_padronizado, kmeans_2_pad, "PC1 padronizada", "PC2 padronizada"),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, (titulo, dados, modelo, xlabel, ylabel) in zip(axes, paineis):
    for grupo, cor in cores_grupo.items():
        no_grupo = modelo.labels_ == grupo
        ax.scatter(dados.loc[no_grupo, "PC1"], dados.loc[no_grupo, "PC2"], color=cor, s=50,
                   marker=marcadores_grupo[grupo], label=f"Grupo {grupo}", **{**ESTILO_PONTOS, "alpha": 0.85})
    ax.scatter(modelo.cluster_centers_[:, 0], modelo.cluster_centers_[:, 1], marker="x", s=160, linewidths=3,
               color=CORES["ocre"], label="Centroide", zorder=3)
    ari = adjusted_rand_score(scores["Behavior"], modelo.labels_)
    ax.set(title=f"{titulo} (ARI = {ari:.2f})", xlabel=xlabel, ylabel=ylabel)
axes[0].legend(fontsize=9)
fig.tight_layout()
plt.show()

## 3.6 Clusterização hierárquica das proteínas

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

# Distância entre proteínas: 1 - |r| (correlação positiva ou negativa = proteínas parecidas)
protein_corr = animal_mean_df[protein_cols].corr()
distancia = 1 - protein_corr.abs()
Z = linkage(squareform(distancia.values, checks=False), method="complete")

grade = sns.clustermap(protein_corr, row_linkage=Z, col_linkage=Z, cmap=CMAP_CORRELACAO, vmin=-1, vmax=1,
                       xticklabels=True, yticklabels=True, figsize=(12, 12),
                       tree_kws={"colors": CORES["texto_secundario"]}, cbar_kws={"label": "Correlação (r)"})
grade.ax_heatmap.tick_params(labelsize=6)
grade.ax_heatmap.set(xlabel="", ylabel="")
plt.show()

In [ ]:
# Corte em t = 0.2: em cada bloco, todos os pares de proteínas têm |r| >= 0.8
blocos = pd.Series(fcluster(Z, t=0.2, criterion="distance"), index=protein_corr.index, name="bloco")
print(f"Blocos: {blocos.nunique()}")

blocos.groupby(blocos).apply(lambda grupo: grupo.index.tolist()).loc[lambda s: s.str.len() > 1]

In [ ]:
# Uma representante por bloco: a proteína mais próxima das demais do mesmo bloco
representantes = [
    distancia.loc[grupo.index, grupo.index].sum(axis=1).idxmin()
    for _, grupo in blocos.groupby(blocos)
]


def cohen_d(a, b):
    a, b = a.dropna(), b.dropna()
    desvio_combinado = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
    return (a.mean() - b.mean()) / desvio_combinado


cs = animal_mean_df["Behavior"] == "C/S"
efeito = pd.Series(
    {p: cohen_d(animal_mean_df.loc[cs, p], animal_mean_df.loc[~cs, p]) for p in representantes},
    name="d de Cohen (C/S - S/C)",
)
efeito = efeito.reindex(efeito.abs().sort_values(ascending=False).index)
efeito.head(10).round(2)

In [ ]:
top_efeito = efeito.head(10).index

fig, axes = plt.subplots(2, 5, figsize=(13, 6.5))
for ax, proteina in zip(axes.flat, top_efeito):
    sns.boxplot(data=animal_mean_df, x="Behavior", y=proteina, order=list(cores_behavior), width=0.5, ax=ax,
                **ESTILO_BOXPLOT)
    sns.stripplot(data=animal_mean_df, x="Behavior", y=proteina, order=list(cores_behavior), hue="Behavior",
                  palette=cores_behavior, legend=False, size=3.5, alpha=0.7, ax=ax)
    ax.set(title=f"{proteina}\nd = {efeito[proteina]:.2f}", xlabel="", ylabel="")
fig.tight_layout()
plt.show()